# rmul-scalar-tensor-mix — ex2: implement __sub__ and __rsub__ so 5 - my_t evaluates to 5 - my_t.value (not my_t.value - 5)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rmul-scalar-tensor-mix`. Running the final beacon cell reports progress against the `PyTorch: __rmul__ scalar/tensor mix` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: __rmul__ scalar/tensor mix` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rmul-scalar-tensor-mix`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rmul-scalar-tensor-mix"
DD_SUBTOPIC = "PyTorch: __rmul__ scalar/tensor mix"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `__sub__` / `__rsub__` — the asymmetry that catches everyone

Ex1 implemented `__mul__` / `__rmul__` — symmetric because multiplication commutes (`2 * t == t * 2`). Subtraction does NOT commute, and that's where `__rsub__` gets misimplemented:

```python
t.__sub__(other)   # returns self - other  (self on the left)
t.__rsub__(other)  # returns other - self  (self on the RIGHT — flipped!)
```

**Why Python calls `__rsub__`.** When you write `5 - my_tensor`, Python first tries `(5).__sub__(my_tensor)` — `int` doesn't know about your tensor class, returns `NotImplemented`. Python then tries the REFLECTED method: `my_tensor.__rsub__(5)`. The convention is that `__rsub__` must compute `other - self`, NOT `self - other`.

**The trap.** Naive implementations write `return self.value - other` for `__rsub__`. That makes `5 - t` equal `t - 5` — exactly backwards.

ARENA's manual-autograd Tensor wrapper hits this trap in chap-0 because MiniTensor needs to mimic torch.Tensor's full op set.

### Exercise 2 — implement __sub__ and __rsub__ so 5 - my_t evaluates to 5 - my_t.value (not my_t.value - 5)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply Python's reflected-op convention to a wrapper class such that `self - other` calls `__sub__(self - other)` and `other - self` calls `__rsub__(other - self)` — preserving operand order in both directions and returning the same wrapper class.
> Keywords: dunder, rsub, operand-order, asymmetry
> ```

**KCs targeted:** `rsub-flips-operand-order`, `wrapper-returns-same-class`

Implement the class `ScalarBox` below. It wraps a single float and supports subtraction with both `int`/`float` and other `ScalarBox` instances. The KEY test: `5 - box(3)` must equal `box(2)` — NOT `box(-2)`.

Requirements:

1. `__init__(self, value)` — store as `self.value = float(value)`.
2. `__sub__(self, other)` — `self - other`. If `other` is a `ScalarBox`, use `other.value`; if a number, use it directly. Returns a new `ScalarBox(self.value - other_v)`.
3. `__rsub__(self, other)` — Python calls this when LEFT operand doesn't know about ScalarBox. Returns `ScalarBox(other - self.value)` — note the operand order is `other - self`, NOT `self - other`.
4. `__eq__(self, other)` — provided for the tests; compare `self.value` to either `other.value` or `other` directly (with tolerance 1e-9).
5. `__repr__` — returns `f'ScalarBox({self.value})'`.

Constraints:
- Both `__sub__` and `__rsub__` must return a new `ScalarBox`, not a plain float.
- `ScalarBox(3) - ScalarBox(1)` (two boxes) goes through `__sub__`, not `__rsub__`.

In [ ]:
class ScalarBox:
    def __init__(self, value):
        self.value = float(value)

    def __sub__(self, other):
        raise NotImplementedError()

    def __rsub__(self, other):
        raise NotImplementedError()

    def __eq__(self, other):
        ov = other.value if isinstance(other, ScalarBox) else other
        return abs(self.value - float(ov)) < 1e-9

    def __hash__(self):
        return hash(self.value)

    def __repr__(self):
        return f'ScalarBox({self.value})'

def ex2_scalarbox():
    return ScalarBox


def _test_ex2():
    Cls = ex2_scalarbox()

    # === box - scalar (normal __sub__) ===
    result = Cls(10) - 3
    assert isinstance(result, Cls), f'must return ScalarBox; got {type(result).__name__}'
    assert result == Cls(7), f'10 - 3 should be 7, got {result}'

    # === scalar - box (the headline __rsub__ test) ===
    result = 5 - Cls(3)
    assert isinstance(result, Cls), f'5 - Cls(3) must return ScalarBox; got {type(result).__name__}'
    assert result == Cls(2), f'5 - Cls(3) should be Cls(2), NOT Cls(-2); got {result}'

    # === The asymmetry: box - scalar != scalar - box ===
    left = Cls(10) - 4    # 10 - 4 = 6
    right = 4 - Cls(10)   # 4 - 10 = -6
    assert left == Cls(6), f'box - scalar wrong: {left}'
    assert right == Cls(-6), f'scalar - box wrong: {right}'
    assert left != right, 'asymmetry must hold: box-scalar != scalar-box'

    # === box - box uses __sub__ (NOT __rsub__) ===
    result = Cls(7) - Cls(2)
    assert isinstance(result, Cls)
    assert result == Cls(5), f'box - box wrong: {result}'

    # === Float scalar on the left ===
    result = 1.5 - Cls(0.5)
    assert result == Cls(1.0), f'1.5 - Cls(0.5) should be Cls(1.0), got {result}'

    # === Negative results work ===
    result = 2 - Cls(5)
    assert result == Cls(-3), f'2 - Cls(5) should be Cls(-3), got {result}'

    # === Zero results work ===
    result = 7 - Cls(7)
    assert result == Cls(0), f'7 - Cls(7) should be Cls(0), got {result}'

    # === Chain: scalar - box - scalar ===
    # 10 - Cls(3) = Cls(7) ; Cls(7) - 2 = Cls(5)
    result = 10 - Cls(3) - 2
    assert result == Cls(5), f'chain wrong: {result}'

    # === Chain: scalar - box - box ===
    # 10 - Cls(3) = Cls(7) ; Cls(7) - Cls(2) = Cls(5)
    result = 10 - Cls(3) - Cls(2)
    assert result == Cls(5), f'mixed chain wrong: {result}'

    # === Repr ===
    assert repr(Cls(3.0)) == 'ScalarBox(3.0)'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class ScalarBox:
    def __init__(self, value):
        self.value = float(value)

    def __sub__(self, other):
        other_v = other.value if isinstance(other, ScalarBox) else other
        return ScalarBox(self.value - other_v)

    def __rsub__(self, other):
        # other is on the LEFT — operand order is other - self, NOT self - other.
        other_v = other.value if isinstance(other, ScalarBox) else other
        return ScalarBox(other_v - self.value)

    def __eq__(self, other):
        ov = other.value if isinstance(other, ScalarBox) else other
        return abs(self.value - float(ov)) < 1e-9

    def __hash__(self):
        return hash(self.value)

    def __repr__(self):
        return f'ScalarBox({self.value})'

def ex2_scalarbox():
    return ScalarBox
```

**The one-line bug.** `__rsub__` returning `self.value - other` (symmetric with `__sub__`) is the universal mistake. It makes `5 - box(3)` equal `box(-2)`. The reflected method MUST flip the operand order: `other - self`.

**Why Python's convention works this way.** Reflected methods are called as a FALLBACK when the LEFT operand returned `NotImplemented`. Python's logic: 'try `left.__op__(right)`, then `right.__rop__(left)`. From `__rop__`'s perspective, `self` is on the right and `other` is on the left — so it must compute `other OP self`, not `self OP other`.

**Same trap for `__rtruediv__`, `__rmod__`, `__rpow__`.** Any non-commutative operator has this asymmetry. Only `+` and `*` (and bitwise `&`, `|`, `^`) commute, so their reflected versions are symmetric — that's why `__rmul__` from ex1 was easy.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()